In [1]:
import pandas as pd
# Remember: library imports are ALWAYS at the top of the script, no exceptions!
import sqlite3
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from math import ceil

In [2]:
df=pd.read_csv('../acdc_dataset.csv', sep=';')

In [3]:
df.head()

,member_id,age,gender,industry,country,is_team_account,n_colleagues_on_account,plan_tier,plan_days_per_week,membership_start,...,uses_event_space,uses_guest_passes,uses_print,uses_mail_handling,uses_phone_booths,uses_day_passes,uses_storage,uses_bike_locker,n_renewals,lifetime_value
0,CW000001,38.0,Female,Information Technology,Portugal,False,0,flex,3,2021-08-01,...,False,True,0,False,True,True,False,False,2,4477.97
1,CW000002,38.0,Male,Information Technology,Portugal,True,2,HOT_DESK,2,2023-11-24,...,False,False,0,True,False,False,False,False,0,165.86
2,CW000003,43.0,Male,Education,DE,True,1,flex,3,2021-12-03,...,False,False,0,False,False,False,False,False,1,4486.17
3,CW000004,38.0,F,Legal,portugal,False,0,flex,3,2022-01-02,...,False,False,0,False,False,False,False,False,0,258.63
4,CW000005,45.0,Female,Information Technology,United Kingdom,False,0,hot_desk,2,2023-10-24,...,False,False,0,False,False,True,False,False,0,335.74


In [4]:
df.shape #rows, columns

(15465, 31)

In [5]:
list(df.columns) #rows, columns

['member_id',
 'age',
 'gender',
 'industry',
 'country',
 'is_team_account',
 'n_colleagues_on_account',
 'plan_tier',
 'plan_days_per_week',
 'membership_start',
 'membership_end',
 'last_badge_swipe',
 'days_since_last_visit',
 'total_visits',
 'avg_weekly_visits',
 'entitled_visits_last_period',
 'actual_visits_last_period',
 'meeting_room_hours',
 'event_space_hours',
 'guest_passes_issued',
 'uses_meeting_rooms',
 'uses_event_space',
 'uses_guest_passes',
 'uses_print',
 'uses_mail_handling',
 'uses_phone_booths',
 'uses_day_passes',
 'uses_storage',
 'uses_bike_locker',
 'n_renewals',
 'lifetime_value']

In [6]:
 #we have null values
info_df = df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15465 entries, 0 to 15464
Data columns (total 31 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   member_id                    15465 non-null  str    
 1   age                          14907 non-null  float64
 2   gender                       14935 non-null  str    
 3   industry                     15465 non-null  str    
 4   country                      15035 non-null  str    
 5   is_team_account              15465 non-null  bool   
 6   n_colleagues_on_account      15465 non-null  int64  
 7   plan_tier                    15360 non-null  str    
 8   plan_days_per_week           15465 non-null  int64  
 9   membership_start             15465 non-null  str    
 10  membership_end               15465 non-null  str    
 11  last_badge_swipe             15465 non-null  str    
 12  days_since_last_visit        15465 non-null  int64  
 13  total_visits               

In [7]:
df.notnull().sum() #summary of whats null and not

member_id                      15465
age                            14907
gender                         14935
industry                       15465
country                        15035
is_team_account                15465
n_colleagues_on_account        15465
plan_tier                      15360
plan_days_per_week             15465
membership_start               15465
membership_end                 15465
last_badge_swipe               15465
days_since_last_visit          15465
total_visits                   15465
avg_weekly_visits              15465
entitled_visits_last_period    15465
actual_visits_last_period      15465
meeting_room_hours             14167
event_space_hours              15465
guest_passes_issued            14194
uses_meeting_rooms             15465
uses_event_space               15465
uses_guest_passes              15465
uses_print                     15465
uses_mail_handling             15465
uses_phone_booths              15465
uses_day_passes                15465
u

In [8]:
null_columns = df.columns[(df.notnull().sum() != 15465)]
list(null_columns) #null columns

['age',
 'gender',
 'country',
 'plan_tier',
 'meeting_room_hours',
 'guest_passes_issued',
 'lifetime_value']

In [9]:
column_percentage = df.isna().mean() * 100 #percentage of missing values
column_percentage = column_percentage[column_percentage > 0] #Gemini for correcting syntax
column_percentage

age                    3.608147
gender                 3.427093
country                2.780472
plan_tier              0.678952
meeting_room_hours     8.393146
guest_passes_issued    8.218558
lifetime_value         6.724863
dtype: float64

In [10]:
null_df_columns = df[null_columns]
null_df_columns #null comlumns

,age,gender,country,plan_tier,meeting_room_hours,guest_passes_issued,lifetime_value
0,38.0,Female,Portugal,flex,0.0,18.0,4477.97
1,38.0,Male,Portugal,HOT_DESK,0.0,0.0,165.86
2,43.0,Male,DE,flex,0.0,0.0,4486.17
3,38.0,F,portugal,flex,0.0,0.0,258.63
4,45.0,Female,United Kingdom,hot_desk,0.0,0.0,335.74
...,...,...,...,...,...,...,...
15460,29.0,Female,Spain,flex,0.0,0.0,1422.36
15461,19.0,Female,Germany,private_office,0.0,0.0,20683.24
15462,28.0,Female,Portugal,hot_desk,0.0,0.0,18.14
15463,32.0,Female,Portugal,dedicated_desk,30.1,0.0,11834.5


In [11]:
null_df_columns.info() #null info

<class 'pandas.DataFrame'>
RangeIndex: 15465 entries, 0 to 15464
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   age                  14907 non-null  float64
 1   gender               14935 non-null  str    
 2   country              15035 non-null  str    
 3   plan_tier            15360 non-null  str    
 4   meeting_room_hours   14167 non-null  float64
 5   guest_passes_issued  14194 non-null  float64
 6   lifetime_value       14425 non-null  str    
dtypes: float64(3), str(4)
memory usage: 1.2 MB


In [12]:
pd.set_option('display.max_columns', 31) #Gemini to look up method to expand df

In [13]:
null_df_age = df[df["age"].isnull()] #Gemini and https://note.nkmk.me/en/python-pandas-nan-extract/ to verify syntax
null_df_age # finding null values

,member_id,age,gender,industry,country,is_team_account,n_colleagues_on_account,plan_tier,plan_days_per_week,membership_start,membership_end,last_badge_swipe,days_since_last_visit,total_visits,avg_weekly_visits,entitled_visits_last_period,actual_visits_last_period,meeting_room_hours,event_space_hours,guest_passes_issued,uses_meeting_rooms,uses_event_space,uses_guest_passes,uses_print,uses_mail_handling,uses_phone_booths,uses_day_passes,uses_storage,uses_bike_locker,n_renewals,lifetime_value
31,CW000030,NaN,Male,Consulting,United Kingdom,False,0,flex,3,2022-10-17,2022-12-30,2022-10-17,75,12,1.11,26.09,10,0.0,0.0,0.0,False,False,False,0,True,False,True,False,False,0,551.5
78,CW000077,NaN,Male,Design & Creative,NaN,False,0,Hot Desk,2,2021-12-12,2023-08-14,2023-08-08,6,163,1.88,17.39,17,0.0,0.0,0.0,False,False,False,0,False,False,False,False,False,0,2402.77
116,CW000112,NaN,Female,Unknown,Portugal,True,4,FLEX,3,2021-09-25,2021-11-08,2021-10-27,12,4,0.68,18.85,5,0.0,0.0,0.0,False,False,False,0,False,False,False,False,False,0,260.13
125,CW000121,NaN,Male,Consulting,Portugal,False,0,flex,3,2022-04-09,1970-01-01,2023-09-05,118,3,0.03,26.09,0,0.0,0.0,3.0,False,False,True,1,True,False,False,False,False,5,NaN
141,CW000137,NaN,Other,Consulting,Portugal,True,1,hot_desk,2,2021-06-11,2023-04-04,2023-02-14,49,48,0.51,17.39,4,48.3,22.6,0.0,True,True,False,0,False,False,False,True,False,2,4750.28
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15323,CW014876,NaN,Female,marketing & pr,Portugal,False,0,dedicated_desk,5,2023-12-21,1970-01-01,2023-12-28,4,7,4.60,7.25,5,0.7,0.4,0.0,True,True,True,1,False,True,True,False,False,5,159.12
15351,CW014903,NaN,Male,Other,Portugal,False,0,full_time,9,2022-07-11,2022-08-13,2022-08-05,8,7,1.54,23.70,8,0.0,0.0,0.0,False,False,False,0,False,False,False,False,False,0,305.2
15376,CW014928,NaN,Female,Legal,Portugal,True,1,full_time,5,2021-01-01,1970-01-01,2024-02-18,999,32,0.20,43.48,2,12.2,0.0,0.0,True,False,False,0,False,False,False,False,False,3,0
15442,CW014994,NaN,Male,Other,Spain,True,3,flex,3,2022-02-08,2022-04-14,2022-02-27,46,9,0.94,26.09,8,0.0,0.0,0.0,False,False,False,0,False,True,False,False,False,0,NaN


In [63]:
df.describe(include="int")

#negative meeting room hours?


,n_colleagues_on_account,plan_days_per_week,days_since_last_visit,total_visits,actual_visits_last_period,uses_print,n_renewals
count,15465.000000,15465.000000,15465.000000,15465.000000,15465.000000,15465.000000,15465.000000
mean,0.537019,3.387262,117.109279,110.406531,13.800970,0.323052,1.005496
std,1.177668,1.458818,275.532807,182.939340,11.387207,0.467658,1.353812
min,-1.000000,0.000000,-956.000000,-112.000000,0.000000,0.000000,-3.000000
25%,0.000000,2.000000,4.000000,8.000000,5.000000,0.000000,0.000000
50%,0.000000,3.000000,13.000000,34.000000,10.000000,0.000000,0.000000
75%,1.000000,5.000000,55.000000,132.000000,20.000000,1.000000,2.000000
max,9.000000,9.000000,1212.000000,6530.000000,60.000000,1.000000,6.000000


In [15]:
df.describe(include="bool")

,is_team_account,uses_meeting_rooms,uses_event_space,uses_guest_passes,uses_mail_handling,uses_phone_booths,uses_day_passes,uses_storage,uses_bike_locker
count,15465,15465,15465,15465,15465,15465,15465,15465,15465
unique,2,2,2,2,2,2,2,2,2
top,False,False,False,False,False,False,False,False,False
freq,11112,10202,13624,12651,11901,11469,12069,13169,15439


In [16]:
df.describe(include="object")

C:\Users\lyric\AppData\Local\Temp\ipykernel_27508\702825166.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.describe(include="object")


,member_id,gender,industry,country,plan_tier,membership_start,membership_end,last_badge_swipe,lifetime_value
count,15465,14935,15465,15035,15360,15465,15465,15465,14425
unique,15450,12,43,33,19,1539,1423,1543,11860
top,CW015101,Female,Information Technology,Portugal,flex,2021-01-01,1970-01-01,2023-12-30,0
freq,2,7414,3322,9959,5182,560,3956,279,1725


In [17]:
df.describe(include="float") #age probably cant be 149
#why is there a 1 year old customer
#guest passes cant be in the negatives
#mean and median in age and avg_weekly_visits seem close
#meeting_room_hours has many negatives. We cant tell if 0 is true median. Same with event_space_hours and guest_passes used.
#Therefore, negatives must be handled before we can know the true mean and median. This also means that the distribution could change once these
#are handled. 
#Most importantly, this means that these values could be skewing the distribution which could alter which models we select and could be creating unecessary
#noise or bigger tails that increase the variance of our model and cause the model to fit poorly. The negative values create invalide outliers that 
#will most likely take a direct toll on the accuracy and fit of our model. 
#Outliers in the age column like 149, would also have a similar effect and need to be handled. 

,age,avg_weekly_visits,entitled_visits_last_period,meeting_room_hours,event_space_hours,guest_passes_issued
count,14907.000000,15465.000000,15465.000000,14167.000000,15465.000000,14194.000000
mean,34.858724,1.777214,27.517393,33.075570,5.117220,6.162956
std,10.394649,1.937515,13.480235,428.169678,26.013097,23.444991
min,1.000000,0.000000,0.000000,-56.600000,-13.500000,-24.000000
25%,29.000000,0.710000,17.390000,0.000000,0.000000,0.000000
50%,34.000000,1.420000,26.090000,0.000000,0.000000,0.000000
75%,40.000000,2.410000,43.480000,7.200000,0.000000,0.000000
max,149.000000,47.400000,78.270000,41220.000000,602.800000,589.000000


In [18]:
df.describe(include='str')

,member_id,gender,industry,country,plan_tier,membership_start,membership_end,last_badge_swipe,lifetime_value
count,15465,14935,15465,15035,15360,15465,15465,15465,14425
unique,15450,12,43,33,19,1539,1423,1543,11860
top,CW015101,Female,Information Technology,Portugal,flex,2021-01-01,1970-01-01,2023-12-30,0
freq,2,7414,3322,9959,5182,560,3956,279,1725


In [19]:
df.dtypes #make float64 for age float8, num colleages can be int4, actual visits can be int8, 
#by looking at the max counts for the numeric columns from the describe() output we can
#see whether a bit dytype is too high.
#ues_print should be bool

member_id                          str
age                            float64
gender                             str
industry                           str
country                            str
is_team_account                   bool
n_colleagues_on_account          int64
plan_tier                          str
plan_days_per_week               int64
membership_start                   str
membership_end                     str
last_badge_swipe                   str
days_since_last_visit            int64
total_visits                     int64
avg_weekly_visits              float64
entitled_visits_last_period    float64
actual_visits_last_period        int64
meeting_room_hours             float64
event_space_hours              float64
guest_passes_issued            float64
uses_meeting_rooms                bool
uses_event_space                  bool
uses_guest_passes                 bool
uses_print                       int64
uses_mail_handling                bool
uses_phone_booths        

In [20]:
testdf = df[df["days_since_last_visit"] == -956.000000]
testdf

,member_id,age,gender,industry,country,is_team_account,n_colleagues_on_account,plan_tier,plan_days_per_week,membership_start,membership_end,last_badge_swipe,days_since_last_visit,total_visits,avg_weekly_visits,entitled_visits_last_period,actual_visits_last_period,meeting_room_hours,event_space_hours,guest_passes_issued,uses_meeting_rooms,uses_event_space,uses_guest_passes,uses_print,uses_mail_handling,uses_phone_booths,uses_day_passes,uses_storage,uses_bike_locker,n_renewals,lifetime_value
13649,CW013250,47.0,Female,Unknown,Portugal,True,1,full_time,5,2021-01-01,2020-12-26,2023-08-09,-956,13,0.08,43.48,1,0.0,0.0,0.0,False,False,False,0,True,False,False,False,False,1,NaN


In [21]:
pd.set_option('display.max_columns', 31) #Gemini to look up method to expand df

## Hypothesis for why days since last visit is negative

In [22]:
testdf #Member's start  and end date is incorrect. End date comes before start date, possibly causing days since last visit to be negative.
#Badge swipe appears to be in 2023, implying that their membership was most likely active until 2023
#We don't yet konw up to what date the csv was updated.
#lifetime_value is NAN

,member_id,age,gender,industry,country,is_team_account,n_colleagues_on_account,plan_tier,plan_days_per_week,membership_start,membership_end,last_badge_swipe,days_since_last_visit,total_visits,avg_weekly_visits,entitled_visits_last_period,actual_visits_last_period,meeting_room_hours,event_space_hours,guest_passes_issued,uses_meeting_rooms,uses_event_space,uses_guest_passes,uses_print,uses_mail_handling,uses_phone_booths,uses_day_passes,uses_storage,uses_bike_locker,n_renewals,lifetime_value
13649,CW013250,47.0,Female,Unknown,Portugal,True,1,full_time,5,2021-01-01,2020-12-26,2023-08-09,-956,13,0.08,43.48,1,0.0,0.0,0.0,False,False,False,0,True,False,False,False,False,1,NaN


In [23]:
df['gender'].unique() #Here we see nan, don't know why 'O' exists.
#Female and Male variations should be merged

<ArrowStringArray>
[ 'Female',    'Male',       'F',   'Other',       nan, ' Female',       'M',
    'male',   ' Male',   'other',  ' Other',  'female',       'O']
Length: 13, dtype: str

In [24]:
df['age'].unique()

array([ 38.,  43.,  45.,  39.,  23.,  33.,  30.,  44.,  37., 131.,  20.,
        40.,  32.,  21.,  29.,  25.,  28.,  51.,  41.,  nan,  31.,  42.,
        34.,  36.,  50.,  24.,  19.,  47.,  54.,  26.,  53.,  27.,  46.,
        18.,  48.,  63.,  35.,  22.,  59.,  49.,   4.,   3.,  58.,  72.,
         9.,   8.,  57.,  52.,  73., 132.,  60.,  55., 144.,  61.,   1.,
         6., 141.,   2.,   7., 137.,  66., 147., 140., 142.,  56.,  62.,
       145.,   5.,  65.,  83.,  14., 138., 136., 146., 134.,  70.,  74.,
        93.,  92., 143., 139.,  91.,  64., 149., 135.])

In [25]:
column_percentage

age                    3.608147
gender                 3.427093
country                2.780472
plan_tier              0.678952
meeting_room_hours     8.393146
guest_passes_issued    8.218558
lifetime_value         6.724863
dtype: float64

In [26]:
df['country'].unique() #normalize values here, spelling is wrong for some countries like Portucal

<ArrowStringArray>
[       'Portugal',              'DE',        'portugal',  'United Kingdom',
           'Spain',         'Germany',     'Netherlands',          'France',
              'PT',  'Other (Europe)',        'Portucal',               nan,
              'UK',         'Ireland',  'United kingdom',              'IE',
       'portugal ',     'Deutschland',         'ireland',         'germany',
          'france',  'united kingdom',  'other (europe)',            'U.K.',
              'GB',              'ES',              'NL',          'España',
          'Espana',     'netherlands',           'spain', 'The Netherlands',
              'FR',         'Holland']
Length: 34, dtype: str

In [27]:
df['plan_tier'].unique() #normalize values

<ArrowStringArray>
[          'flex',       'HOT_DESK',       'hot_desk', 'dedicated_desk',
      'full_time', 'private_office',       'Hot Desk',  'dedicateddesk',
              nan,      'FULL_TIME',        'hotdesk',           'Flex',
           'FLEX',       'fulltime',  'privateoffice',      'Full Time',
 'DEDICATED_DESK', 'Dedicated Desk', 'Private Office', 'PRIVATE_OFFICE']
Length: 20, dtype: str

In [28]:
df['meeting_room_hours'].unique() 

array([  0. , 129. ,  42.1, ..., 166.4,  48.8, 183. ], shape=(1773,))

In [29]:
df['guest_passes_issued'].unique() #negative values

array([ 18.,   0.,   1.,  37.,  nan,  23.,  57.,  17.,  14.,   2.,   5.,
        28., 189.,  34.,  30.,  35.,  12.,   3.,  19.,  10.,  55.,  58.,
        61.,  91.,   4.,  24., 100.,  15.,  13.,  43.,  54., 135.,  68.,
        65.,  16.,  25.,  74.,  20.,  45.,  40.,  46.,   9., 200.,  52.,
        72.,  26.,   8.,  78.,   6., 132.,   7.,  85.,  49.,  84.,  22.,
        64.,  21.,  71.,  27.,  48.,  62.,  77.,  44., 101.,  53., 116.,
       147., 127.,  60.,  32., 228.,  93.,  33.,  36.,  42.,  31.,  51.,
       144.,  47., 142., 137., 150.,  56., 118., 110.,  38.,  67.,  75.,
       247.,  82.,  39., 274., 168., 226., 112.,  11., 374., 143., 109.,
        70.,  83.,  99.,  79., 136.,  41., 146., 133.,  63.,  88.,  -2.,
       348., 108.,  95., 138., 186.,  73.,  50., 230., 171., 111.,  86.,
       115.,  29.,  97., 123., 154., 139., 196., 107., 149., 102.,  94.,
        69., 219.,  66., 119.,  90., 140., 187., 257., 117.,  81., 152.,
       251., 173., 114., 129.,  98., 153., 179.,  8

In [30]:
df['lifetime_value'].unique() #negative values

<ArrowStringArray>
[ '4477.97',   '165.86',  '4486.17',   '258.63',   '335.74',   '413.17',
   '257.95',   '4636.8',   '281.37',  '8741.88',
 ...
  '3577.95', '26549.23',  '14911.6',  '1559.46',   '344.69',   '190.85',
   '231.28',  '2031.02',   '784.69',  '8358.39']
Length: 11861, dtype: str

In [35]:
column_percentage = df.isna().mean() * 100 #percentage of missing values

,member_id,age,gender,industry,country,is_team_account,n_colleagues_on_account,plan_tier,plan_days_per_week,membership_start,membership_end,last_badge_swipe,days_since_last_visit,total_visits,avg_weekly_visits,entitled_visits_last_period,actual_visits_last_period,meeting_room_hours,event_space_hours,guest_passes_issued,uses_meeting_rooms,uses_event_space,uses_guest_passes,uses_print,uses_mail_handling,uses_phone_booths,uses_day_passes,uses_storage,uses_bike_locker,n_renewals,lifetime_value


In [41]:
numeric_columns = df.select_dtypes(include='number')  #percentage of missing values
negative_numerics_percentage = (numeric_columns < 0).mean() * 100
negative_numerics_percentage

age                            0.000000
n_colleagues_on_account        3.659877
plan_days_per_week             0.000000
days_since_last_visit          0.362108
total_visits                   0.271581
avg_weekly_visits              0.000000
entitled_visits_last_period    0.000000
actual_visits_last_period      0.000000
meeting_room_hours             0.103459
event_space_hours              0.051730
guest_passes_issued            0.045263
uses_print                     0.000000
n_renewals                     0.426770
dtype: float64

In [46]:
type(numeric_columns.columns)

pandas.Index

In [62]:
negative_columns = numeric_columns.columns[(numeric_columns < 0).any()]
list(negative_columns)

['n_colleagues_on_account',
 'days_since_last_visit',
 'total_visits',
 'meeting_room_hours',
 'event_space_hours',
 'guest_passes_issued',
 'n_renewals']

In [ ]:
negative_columns = numeric_columns.columns[(numeric_columns < 0).any()]
